# Ensemble methods. Exercises


In this section we have only two exercise:

1. Find the best three classifier in the stacking method using the classifiers from scikit-learn package.

2. Build arcing arc-x4 method.

In [74]:
import pickle
with open("data_set.pkl", "rb") as f:
    data_set = pickle.load(f)
with open("labels.pkl", "rb") as f:
    labels = pickle.load(f)
with open("test_data_set.pkl", "rb") as f:
    test_data_set = pickle.load(f)
with open("test_labels.pkl", "rb") as f:
    test_labels = pickle.load(f)
with open("unique_labels.pkl", "rb") as f:
    unique_labels = pickle.load(f)

## Exercise 1: Find the best three classifier in the stacking method

Please use the following classifiers:

* Linear regression,
* Nearest Neighbors,
* Linear SVM,
* Decision Tree,
* Naive Bayes,
* QDA.

In [76]:
import numpy as np
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

In [ ]:
def build_classifiers():
    model_pool = [
        ("Linear Regression", LinearRegression()),
        ("Nearest Neighbors", KNeighborsClassifier()),
        ("Linear SVM", SVC(kernel='linear')),
        ("Decision Tree", DecisionTreeClassifier(random_state=0)),
        ("Naive Bayes", GaussianNB()),
        ("QDA", QuadraticDiscriminantAnalysis())
    ]

    ranking = []
    for triplet in combinations(model_pool, 3):
        trained_models = []
        single_scores = []

        for _, model in triplet:
            model.fit(data_set, labels)
            pred = model.predict(test_data_set)
            if isinstance(model, LinearRegression):
                pred = np.rint(pred).astype(labels.dtype)
            single_scores.append(accuracy_score(test_labels, pred))
            trained_models.append(model)

        group_score = float(np.mean(single_scores))
        ranking.append((group_score, triplet, trained_models))

    ranking.sort(key=lambda x: x[0], reverse=True)

    print("Top 10 avg accuracy:")
    for idx, (score, triplet, _) in enumerate(ranking[:10], 1):
        names = [name for name, _ in triplet]
        print(f"{idx}. {score:.4f} -> {names}")

    return ranking[0][2]

In [81]:
def build_stacked_classifier(classifiers):
    output = []
    for classifier in classifiers:
        output.append(classifier.predict(data_set))
    output = np.array(output).reshape((130,3))

    # stacked classifier part:
    stacked_classifier = DecisionTreeClassifier(random_state=0)
    stacked_classifier.fit(output.reshape((130,3)), labels.reshape((130,)))
    test_set = []
    for classifier in classifiers:
        test_set.append(classifier.predict(test_data_set))
    test_set = np.array(test_set).reshape((len(test_set[0]),3))
    predicted = stacked_classifier.predict(test_set)
    return predicted

In [82]:
classifiers = build_classifiers()
predicted = build_stacked_classifier(classifiers)
accuracy = accuracy_score(test_labels, predicted)

Top 10 avg accuracy:
1. 0.9500 -> ['Linear Regression', 'Nearest Neighbors', 'QDA']
2. 0.9500 -> ['Nearest Neighbors', 'Linear SVM', 'QDA']
3. 0.9500 -> ['Nearest Neighbors', 'Decision Tree', 'QDA']
4. 0.9500 -> ['Nearest Neighbors', 'Naive Bayes', 'QDA']
5. 0.9333 -> ['Linear Regression', 'Nearest Neighbors', 'Linear SVM']
6. 0.9333 -> ['Linear Regression', 'Nearest Neighbors', 'Decision Tree']
7. 0.9333 -> ['Linear Regression', 'Nearest Neighbors', 'Naive Bayes']
8. 0.9333 -> ['Nearest Neighbors', 'Linear SVM', 'Decision Tree']
9. 0.9333 -> ['Nearest Neighbors', 'Linear SVM', 'Naive Bayes']
10. 0.9333 -> ['Nearest Neighbors', 'Decision Tree', 'Naive Bayes']


## Exercise 2:

Use the boosting method and change the code to fullfilt the following requirements:

* the weights should be calculated as:
$w_{n}^{(t+1)}=\frac{1+ I(y_{n}\neq h_{t}(x_{n})}{\sum_{i=1}^{N}1+I(y_{n}\neq h_{t}(x_{n})}$,
* the prediction is done with a voting method.

In [107]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# prepare data set

def generate_data(sample_number, feature_number, label_number):
    data_set = np.random.random_sample((sample_number, feature_number))
    labels = np.random.choice(label_number, sample_number)
    return data_set, labels

labels = 2
dimension = 2
test_set_size = 1000
train_set_size = 5000
train_set, train_labels = generate_data(train_set_size, dimension, labels)
test_set, test_labels = generate_data(test_set_size, dimension, labels)

# init weights
number_of_iterations = 10
weights = np.ones((test_set_size,)) / test_set_size


def train_model(classifier, weights):
    return classifier.fit(X=test_set, y=test_labels, sample_weight=weights)

def calculate_error(model):
    predicted = model.predict(test_set)
    I=calculate_accuracy_vector(predicted, test_labels)
    Z=np.sum(I)
    return (1+Z)/1.0

Fill the two functions below:

In [106]:
def set_new_weights(model):
    # fill the code here (two lines)
    predicted = (model.predict(test_set) != test_labels).astype(int)
    return (1 + predicted) / np.sum(1 + predicted)

Train the classifier with the code below:

In [108]:
classifier = DecisionTreeClassifier(max_depth=1, random_state=1)
classifier.fit(X=train_set, y=train_labels)
alphas = []
classifiers = []
for iteration in range(number_of_iterations):
    model = train_model(classifier, weights)
    weights = set_new_weights(model)
    classifiers.append(model)

print(weights)


validate_x, validate_label = generate_data(1, dimension, labels)

[0.00133511 0.00133511 0.00066756 0.00066756 0.00066756 0.00133511
 0.00066756 0.00066756 0.00133511 0.00133511 0.00133511 0.00066756
 0.00133511 0.00066756 0.00133511 0.00066756 0.00066756 0.00133511
 0.00133511 0.00066756 0.00066756 0.00066756 0.00066756 0.00066756
 0.00066756 0.00133511 0.00066756 0.00066756 0.00133511 0.00133511
 0.00133511 0.00066756 0.00133511 0.00066756 0.00133511 0.00066756
 0.00066756 0.00133511 0.00133511 0.00133511 0.00133511 0.00133511
 0.00066756 0.00066756 0.00133511 0.00066756 0.00133511 0.00066756
 0.00133511 0.00066756 0.00066756 0.00066756 0.00066756 0.00066756
 0.00133511 0.00066756 0.00066756 0.00133511 0.00066756 0.00133511
 0.00066756 0.00066756 0.00133511 0.00066756 0.00066756 0.00066756
 0.00133511 0.00066756 0.00133511 0.00133511 0.00133511 0.00133511
 0.00133511 0.00133511 0.00066756 0.00066756 0.00066756 0.00066756
 0.00066756 0.00133511 0.00066756 0.00066756 0.00066756 0.00133511
 0.00133511 0.00066756 0.00066756 0.00133511 0.00133511 0.0013

Set the validation data set:

In [103]:
validate_x, validate_label = generate_data(1, dimension, labels)

Fill the prediction code:

In [116]:
def get_prediction(x):
    votes = np.array([model.predict(x) for model in classifiers])
    final_prediction = []
    for sample_votes in votes.T:
        final_prediction.append(np.bincount(sample_votes.astype(int)).argmax())
    return np.array(final_prediction)

Test it:

In [118]:
prediction = get_prediction(validate_x)[0]

print("Predykcja:", prediction)

Predykcja: 0
